# Tahap 8 — Labeling Risiko Banjir

Notebook ini mengimplementasikan seluruh proses Tahap 8 (Labeling Risiko Banjir) sesuai metodologi pada BAB III Subbab 3.7.3 (Pembentukan Label Risiko Banjir).

**Input:** `07_missing_handled.csv`
**Output:**
- `08_labeled_dataset.csv`
- `STAGE8_REPORT.md`

Notebook bersifat reproducible: menjalankan seluruh cell dari awal menggunakan file input akan menghasilkan output yang identik.

**Aturan Labeling:**
- Label dasar berdasarkan RR (Tabel 3.3 BAB III)
- Boundary zone operasional: 15–25 mm, 45–55 mm, 95–105 mm
- Jika RR berada pada boundary zone dan CAPE > 2500 J/kg → naikkan label satu tingkat (Aturan R3)
- Jika RR berada pada boundary zone dan CAPE ≤ 2500 J/kg → label tetap mengikuti RR (Aturan R2)
- Jika CAPE NaN (data sounding tidak tersedia) → label tetap mengikuti RR
- Tidak ada baris yang dihapus, tidak ada imputasi/scaling/balancing tambahan


## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 2. Load Dataset

Path input dan output diatur relatif agar notebook dapat dijalankan ulang dari direktori mana pun, selama struktur folder `uploads/` dan `outputs/` tersedia relatif terhadap notebook, atau sesuaikan `INPUT_PATH` di bawah.

In [2]:
# Sesuaikan path berikut jika struktur folder berbeda
INPUT_PATH = '/mnt/user-data/uploads/07_missing_handled.csv'
OUTPUT_DIR = '/mnt/user-data/outputs'
OUTPUT_CSV_PATH = os.path.join(OUTPUT_DIR, '08_labeled_dataset.csv')
OUTPUT_REPORT_PATH = os.path.join(OUTPUT_DIR, 'STAGE8_REPORT.md')

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(INPUT_PATH)
n_input = len(df)

print("Shape:", df.shape)
df.head()


Shape: (2922, 13)


,date,selected_hour,selection_status,rr,tavg,rh,cin,kindex,li,tt,sweat,cape,missing_group
0,2017-01-01,12Z,SELECTED,26.9,28.2,83.9,-4.906,34.4,-4.510,42.2,219.162,2226.401,IMPUTED_OGIMET
1,2017-01-02,12Z,SELECTED,26.9,26.5,87.4,-22.887,35.2,-2.955,41.5,223.381,921.164,ORIGINAL
2,2017-01-03,12Z,SELECTED,78.0,26.2,87.6,0.000,36.4,-3.317,40.5,277.368,1671.572,ORIGINAL
3,2017-01-04,12Z,SELECTED,33.0,24.7,92.4,-4.865,35.1,-0.752,39.0,294.560,397.409,ORIGINAL
4,2017-01-05,12Z,SELECTED,83.0,25.7,88.3,-15.811,38.0,-5.124,44.1,299.131,2064.721,ORIGINAL


## 3. Pre-Label Validation

Sebelum labeling dilakukan, dataset diaudit terlebih dahulu:
- Struktur dataset (jumlah baris/kolom, tipe data)
- Ketersediaan kolom yang dibutuhkan (`rr`, `cape`)
- Audit missing value pada RR dan CAPE
- Audit distribusi RR
- Audit jumlah record pada setiap boundary zone

Jika ditemukan masalah struktural, proses dihentikan sebelum melangkah ke labeling.

In [3]:
print("=== STRUKTUR DATASET ===")
print("Jumlah baris :", df.shape[0])
print("Jumlah kolom :", df.shape[1])
print("Kolom        :", df.columns.tolist())
print()
print(df.dtypes)


=== STRUKTUR DATASET ===
Jumlah baris : 2922
Jumlah kolom : 13
Kolom        : ['date', 'selected_hour', 'selection_status', 'rr', 'tavg', 'rh', 'cin', 'kindex', 'li', 'tt', 'sweat', 'cape', 'missing_group']

date                    str
selected_hour           str
selection_status        str
rr                  float64
tavg                float64
rh                  float64
cin                 float64
kindex              float64
li                  float64
tt                  float64
sweat               float64
cape                float64
missing_group           str
dtype: object


In [4]:
required_cols = ['rr', 'cape']
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise SystemExit(f"STOP: kolom wajib tidak ditemukan: {missing_cols}")
else:
    print("OK: seluruh kolom yang dibutuhkan (rr, cape) tersedia.")


OK: seluruh kolom yang dibutuhkan (rr, cape) tersedia.


In [5]:
print("=== AUDIT MISSING VALUE ===")
print(df.isna().sum())
print()
print("Missing value pada rr   :", df['rr'].isna().sum())
print("Missing value pada cape :", df['cape'].isna().sum())
print()
print("Duplikasi tanggal:", df['date'].duplicated().sum())
print("Rentang tanggal  :", df['date'].min(), "s.d.", df['date'].max())


=== AUDIT MISSING VALUE ===
date                  0
selected_hour         0
selection_status      0
rr                    0
tavg                  0
rh                    0
cin                 148
kindex              148
li                  148
tt                  148
sweat               148
cape                148
missing_group         0
dtype: int64

Missing value pada rr   : 0
Missing value pada cape : 148

Duplikasi tanggal: 0
Rentang tanggal  : 2017-01-01 s.d. 2024-12-31


In [6]:
print("=== AUDIT KESELARASAN MISSING CAPE ===")
if 'missing_group' in df.columns:
    print(df.loc[df['cape'].isna(), 'missing_group'].value_counts())
if 'selection_status' in df.columns:
    print()
    print(df.loc[df['cape'].isna(), 'selection_status'].value_counts())


=== AUDIT KESELARASAN MISSING CAPE ===
missing_group
NO_SOUNDING    148
Name: count, dtype: int64

selection_status
NO_SOUNDING    148
Name: count, dtype: int64


In [7]:
print("=== AUDIT RR ===")
print(df['rr'].describe())
print()
print("Nilai rr negatif  :", (df['rr'] < 0).sum())
print("Nilai rr NaN      :", df['rr'].isna().sum())


=== AUDIT RR ===
count    2922.000000
mean       12.435216
std        25.101617
min         0.000000
25%         0.000000
50%         1.600000
75%        13.275000
max       355.300000
Name: rr, dtype: float64

Nilai rr negatif  : 0
Nilai rr NaN      : 0


In [8]:
print("=== AUDIT CAPE ===")
print(df['cape'].describe())
print()
print("Nilai cape negatif :", (df['cape'] < 0).sum())
print("Nilai cape NaN     :", df['cape'].isna().sum())


=== AUDIT CAPE ===
count     2774.000000
mean      1529.052656
std       1315.196757
min          0.000000
25%        565.670500
50%       1504.867000
75%       2291.457500
max      38045.450000
Name: cape, dtype: float64

Nilai cape negatif : 0
Nilai cape NaN     : 148


In [9]:
def base_label(rr):
    if rr < 20:
        return "Rendah"
    elif rr < 50:
        return "Sedang"
    elif rr < 100:
        return "Tinggi"
    else:
        return "Sangat Tinggi"

print("=== DISTRIBUSI KATEGORI DASAR (RR SAJA, SEBELUM PENYESUAIAN CAPE) ===")
base_preview = df['rr'].apply(base_label)
print(base_preview.value_counts())


=== DISTRIBUSI KATEGORI DASAR (RR SAJA, SEBELUM PENYESUAIAN CAPE) ===
rr
Rendah           2341
Sedang            375
Tinggi            162
Sangat Tinggi      44
Name: count, dtype: int64


In [10]:
def in_boundary(rr):
    return (15 <= rr <= 25) or (45 <= rr <= 55) or (95 <= rr <= 105)

boundary_mask = df['rr'].apply(in_boundary)

def zone(rr):
    if 15 <= rr <= 25:
        return "15-25"
    if 45 <= rr <= 55:
        return "45-55"
    if 95 <= rr <= 105:
        return "95-105"
    return None

zones = df['rr'].apply(zone)

print("=== JUMLAH RECORD PADA SETIAP BOUNDARY ZONE ===")
print(zones.value_counts(dropna=True))
print()
print("Total record pada boundary zone:", boundary_mask.sum())


=== JUMLAH RECORD PADA SETIAP BOUNDARY ZONE ===
rr
15-25     214
45-55      64
95-105     11
Name: count, dtype: int64

Total record pada boundary zone: 289


In [11]:
print("=== DISTRIBUSI CAPE DALAM BOUNDARY ZONE ===")
for z in ["15-25", "45-55", "95-105"]:
    sub = df[zones == z]
    print(f"--- Zona {z} mm ---")
    print("  count       :", len(sub))
    print("  CAPE > 2500 :", (sub['cape'] > 2500).sum())
    print("  CAPE <= 2500:", (sub['cape'] <= 2500).sum())
    print("  CAPE NaN    :", sub['cape'].isna().sum())


=== DISTRIBUSI CAPE DALAM BOUNDARY ZONE ===
--- Zona 15-25 mm ---
  count       : 214
  CAPE > 2500 : 22
  CAPE <= 2500: 177
  CAPE NaN    : 15
--- Zona 45-55 mm ---
  count       : 64
  CAPE > 2500 : 2
  CAPE <= 2500: 60
  CAPE NaN    : 2
--- Zona 95-105 mm ---
  count       : 11
  CAPE > 2500 : 0
  CAPE <= 2500: 11
  CAPE NaN    : 0


In [12]:
# === VALIDATION GATE ===
# Proses dihentikan jika ditemukan masalah struktural.
validation_passed = True
validation_messages = []

if df.shape[0] == 0:
    validation_passed = False
    validation_messages.append("Dataset kosong.")

if missing_cols:
    validation_passed = False
    validation_messages.append(f"Kolom wajib tidak ditemukan: {missing_cols}")

if df['rr'].isna().sum() > 0:
    validation_passed = False
    validation_messages.append("Ditemukan missing value pada kolom rr yang tidak dapat diproses tanpa penanganan lebih lanjut.")

if (df['rr'] < 0).sum() > 0:
    validation_passed = False
    validation_messages.append("Ditemukan nilai rr negatif (anomali fisik).")

if (df['cape'] < 0).sum() > 0:
    validation_passed = False
    validation_messages.append("Ditemukan nilai cape negatif (anomali fisik).")

if df['date'].duplicated().sum() > 0:
    validation_passed = False
    validation_messages.append("Ditemukan duplikasi tanggal.")

if validation_passed:
    print("VALIDASI: PASS — dataset valid, lanjut ke tahap labeling.")
else:
    print("VALIDASI: FAIL")
    for m in validation_messages:
        print(" -", m)
    raise SystemExit("STOP: proses dihentikan karena ditemukan masalah struktural.")


VALIDASI: PASS — dataset valid, lanjut ke tahap labeling.


## 4. Implementasi Aturan Labeling

Aturan keputusan (Decision Rules, Tabel 3.5 BAB III):

| Aturan | Kondisi | Kategori Risiko |
|---|---|---|
| R1 | RR di luar boundary zone | Mengikuti kategori RR |
| R2 | RR pada boundary zone, CAPE rendah/sedang/tidak tersedia | Mengikuti kategori RR |
| R3 | RR pada boundary zone, CAPE tinggi (> 2500) | Naik satu tingkat |
| R4 | RR sudah Sangat Tinggi | Tetap Sangat Tinggi |

Boundary zone operasional yang digunakan: 15–25 mm, 45–55 mm, 95–105 mm.

In [13]:
LEVELS = ["Rendah", "Sedang", "Tinggi", "Sangat Tinggi"]

def upgrade(label):
    idx = LEVELS.index(label)
    return LEVELS[min(idx + 1, len(LEVELS) - 1)]

def compute_label(row):
    rr = row['rr']
    cape = row['cape']
    base = base_label(rr)
    boundary = in_boundary(rr)

    if not boundary:
        # R1
        return pd.Series([base, "RR_ONLY"])

    if pd.isna(cape):
        # R2 (CAPE tidak tersedia)
        return pd.Series([base, "RR_ONLY_CAPE_MISSING"])

    if cape > 2500:
        # R3
        return pd.Series([upgrade(base), "RR_CAPE_MODIFIED"])
    else:
        # R2 (CAPE rendah/sedang)
        return pd.Series([base, "RR_ONLY"])


## 5. Pembuatan `risk_label` dan `label_source`

In [14]:
df[['risk_label', 'label_source']] = df.apply(compute_label, axis=1)

n_output = len(df)

print("Kolom risk_label dan label_source berhasil ditambahkan.")
df[['date', 'rr', 'cape', 'risk_label', 'label_source']].head(10)


Kolom risk_label dan label_source berhasil ditambahkan.


,date,rr,cape,risk_label,label_source
0,2017-01-01,26.9,2226.401,Sedang,RR_ONLY
1,2017-01-02,26.9,921.164,Sedang,RR_ONLY
2,2017-01-03,78.0,1671.572,Tinggi,RR_ONLY
3,2017-01-04,33.0,397.409,Sedang,RR_ONLY
4,2017-01-05,83.0,2064.721,Tinggi,RR_ONLY
5,2017-01-06,57.0,198.461,Tinggi,RR_ONLY
6,2017-01-07,9.0,2168.771,Rendah,RR_ONLY
7,2017-01-08,5.0,1713.096,Rendah,RR_ONLY
8,2017-01-09,0.5,601.610,Rendah,RR_ONLY
9,2017-01-10,0.0,551.840,Rendah,RR_ONLY


## 6. Audit Distribusi `risk_label`

In [15]:
risk_label_counts = df['risk_label'].value_counts()
risk_label_pct = (df['risk_label'].value_counts(normalize=True) * 100).round(2)

risk_label_summary = pd.DataFrame({
    'jumlah': risk_label_counts,
    'persentase': risk_label_pct
})
risk_label_summary


,jumlah,persentase
risk_label,,
Rendah,2331,79.77
Sedang,372,12.73
Tinggi,174,5.95
Sangat Tinggi,45,1.54


## 7. Audit Distribusi `label_source`

In [16]:
label_source_counts = df['label_source'].value_counts()
label_source_pct = (df['label_source'].value_counts(normalize=True) * 100).round(2)

label_source_summary = pd.DataFrame({
    'jumlah': label_source_counts,
    'persentase': label_source_pct
})
label_source_summary


,jumlah,persentase
label_source,,
RR_ONLY,2881,98.60
RR_CAPE_MODIFIED,24,0.82
RR_ONLY_CAPE_MISSING,17,0.58


## 8. Audit CAPE Modification

In [17]:
modified = df[df['label_source'] == 'RR_CAPE_MODIFIED'].copy()
n_modified = len(modified)

print("Jumlah record yang dimodifikasi oleh CAPE:", n_modified)

# rincian per zona
modified_zone = modified['rr'].apply(zone)
print()
print("Rincian modifikasi per boundary zone:")
print(modified_zone.value_counts())

# arah kenaikan label
base_before = modified['rr'].apply(base_label)
direction = base_before + " -> " + modified['risk_label']
print()
print("Arah kenaikan label:")
print(direction.value_counts())


Jumlah record yang dimodifikasi oleh CAPE: 24

Rincian modifikasi per boundary zone:
rr
15-25    22
45-55     2
Name: count, dtype: int64

Arah kenaikan label:
Sedang -> Tinggi           13
Rendah -> Sedang           10
Tinggi -> Sangat Tinggi     1
Name: count, dtype: int64


In [18]:
display_cols = ['date', 'rr', 'cape', 'risk_label', 'label_source']
modified_examples = modified[display_cols].reset_index(drop=True)
print(f"Menampilkan seluruh {len(modified_examples)} record yang termodifikasi "
      f"(minimum 20 contoh dipenuhi karena total termodifikasi = {len(modified_examples)}):")
modified_examples


Menampilkan seluruh 24 record yang termodifikasi (minimum 20 contoh dipenuhi karena total termodifikasi = 24):


,date,rr,cape,risk_label,label_source
0,2017-04-14,21.0,2612.556,Tinggi,RR_CAPE_MODIFIED
1,2017-09-26,18.0,2540.856,Sedang,RR_CAPE_MODIFIED
2,2017-11-02,17.2,2734.026,Sedang,RR_CAPE_MODIFIED
3,2018-05-07,25.0,4879.221,Tinggi,RR_CAPE_MODIFIED
4,2018-05-08,15.0,5318.834,Sedang,RR_CAPE_MODIFIED
5,2018-11-04,24.0,2642.059,Tinggi,RR_CAPE_MODIFIED
6,2019-02-17,23.0,2754.690,Tinggi,RR_CAPE_MODIFIED
7,2019-03-23,25.0,2660.591,Tinggi,RR_CAPE_MODIFIED
8,2020-04-23,19.8,2741.798,Sedang,RR_CAPE_MODIFIED
9,2020-10-23,22.0,2578.076,Tinggi,RR_CAPE_MODIFIED


## 9. Validasi Integritas Data

In [19]:
integrity = {
    'row_input': n_input,
    'row_output': n_output,
    'selisih_baris': n_input - n_output,
    'risk_label_kosong': int(df['risk_label'].isna().sum()),
    'label_source_kosong': int(df['label_source'].isna().sum()),
}

for k, v in integrity.items():
    print(f"{k:22s}: {v}")

assert integrity['row_input'] == integrity['row_output'], "Row input dan output tidak sama!"
assert integrity['risk_label_kosong'] == 0, "Ditemukan risk_label kosong!"
assert integrity['label_source_kosong'] == 0, "Ditemukan label_source kosong!"

print()
print("Validasi integritas: PASS")


row_input             : 2922
row_output            : 2922
selisih_baris         : 0
risk_label_kosong     : 0
label_source_kosong   : 0

Validasi integritas: PASS


## 10. Export CSV

In [20]:
df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Dataset berlabel disimpan ke: {OUTPUT_CSV_PATH}")
print("Shape akhir:", df.shape)


Dataset berlabel disimpan ke: /mnt/user-data/outputs/08_labeled_dataset.csv
Shape akhir: (2922, 15)


## 11. Generate Report (`STAGE8_REPORT.md`)

In [21]:
def fmt_int(x):
    return f"{x:,}".replace(",", ".")

boundary_total = int(boundary_mask.sum())
zone_counts = zones.value_counts()
zone_1525 = int(zone_counts.get("15-25", 0))
zone_4555 = int(zone_counts.get("45-55", 0))
zone_95105 = int(zone_counts.get("95-105", 0))

def zone_cape_stats(z):
    sub = df[zones == z]
    return {
        "total": len(sub),
        "gt2500": int((sub['cape'] > 2500).sum()),
        "le2500": int((sub['cape'] <= 2500).sum()),
        "nan": int(sub['cape'].isna().sum()),
    }

stats_1525 = zone_cape_stats("15-25")
stats_4555 = zone_cape_stats("45-55")
stats_95105 = zone_cape_stats("95-105")

base_counts = base_preview.value_counts()

rl_rows = "\n".join(
    f"| {idx} | {fmt_int(risk_label_summary.loc[idx, 'jumlah'])} | {risk_label_summary.loc[idx, 'persentase']}% |"
    for idx in ["Rendah", "Sedang", "Tinggi", "Sangat Tinggi"]
    if idx in risk_label_summary.index
)

ls_rows = "\n".join(
    f"| {idx} | {fmt_int(label_source_summary.loc[idx, 'jumlah'])} | {label_source_summary.loc[idx, 'persentase']}% |"
    for idx in ["RR_ONLY", "RR_CAPE_MODIFIED", "RR_ONLY_CAPE_MISSING"]
    if idx in label_source_summary.index
)

direction_counts = direction.value_counts()
_direction_order = ["Rendah -> Sedang", "Sedang -> Tinggi", "Tinggi -> Sangat Tinggi"]
direction_lines = "\n".join(
    f"- {k}: {int(direction_counts.get(k, 0))} record"
    for k in _direction_order
    if k in direction_counts.index
)

example_rows = "\n".join(
    f"| {i+1} | {r['date']} | {r['rr']} | {r['cape']:.3f} | {r['risk_label']} | {r['label_source']} |"
    for i, r in modified_examples.iterrows()
)

n_examples = len(modified_examples)
if n_examples <= 20:
    example_note = (
        f"Catatan: total record yang termodifikasi hanya berjumlah {n_examples} "
        f"(kurang dari atau sama dengan 20 diminta sebagai minimum, sehingga seluruh "
        f"{n_examples} record ditampilkan secara lengkap)."
    )
else:
    example_note = f"Menampilkan seluruh {n_examples} record yang termodifikasi (melebihi minimum 20 contoh yang diminta)."

report = f"""# STAGE 8 REPORT — Labeling Risiko Banjir

**Input:** `07_missing_handled.csv` ({fmt_int(n_input)} baris, {df.shape[1]-2} kolom awal)
**Output:** `08_labeled_dataset.csv` ({fmt_int(n_output)} baris, {df.shape[1]} kolom)
**Sumber metodologi:** BAB III — Subbab 3.7.3 (Pembentukan Label Risiko Banjir), Tabel 3.3-3.5

---

## 1. Ringkasan Hasil Validasi Input (Pre-Label Validation)

| Pemeriksaan | Hasil | Status |
|---|---|---|
| Jumlah baris | {fmt_int(n_input)} | OK |
| Jumlah kolom (input) | {df.shape[1]-2} | OK |
| Kolom `rr` tersedia | Ya (float64) | OK |
| Kolom `cape` tersedia | Ya (float64) | OK |
| Duplikasi tanggal | {int(df['date'].duplicated().sum())} | OK |
| Rentang tanggal | {df['date'].min()} s.d. {df['date'].max()}, tanpa gap tanggal | OK |
| Missing value pada `rr` | {int(df['rr'].isna().sum())} | OK |
| Missing value pada `cape` | {int(df['cape'].isna().sum())} ({(df['cape'].isna().sum()/n_input*100):.2f}%) - seluruhnya berasal dari baris `NO_SOUNDING` | OK (ditangani via `RR_ONLY_CAPE_MISSING`) |
| Nilai `rr` negatif | {int((df['rr']<0).sum())} | OK |
| Nilai `cape` negatif | {int((df['cape']<0).sum())} | OK |

**Distribusi RR terhadap kategori dasar (sebelum penyesuaian CAPE):**

| Kategori dasar (RR) | Jumlah |
|---|---|
| Rendah (RR < 20) | {fmt_int(int(base_counts.get('Rendah', 0)))} |
| Sedang (20 <= RR < 50) | {fmt_int(int(base_counts.get('Sedang', 0)))} |
| Tinggi (50 <= RR < 100) | {fmt_int(int(base_counts.get('Tinggi', 0)))} |
| Sangat Tinggi (RR >= 100) | {fmt_int(int(base_counts.get('Sangat Tinggi', 0)))} |

**Jumlah record pada setiap boundary zone (parameter operasional):**

| Boundary Zone | Jumlah Record | CAPE > 2500 | CAPE <= 2500 | CAPE NaN |
|---|---|---|---|---|
| 15-25 mm | {stats_1525['total']} | {stats_1525['gt2500']} | {stats_1525['le2500']} | {stats_1525['nan']} |
| 45-55 mm | {stats_4555['total']} | {stats_4555['gt2500']} | {stats_4555['le2500']} | {stats_4555['nan']} |
| 95-105 mm | {stats_95105['total']} | {stats_95105['gt2500']} | {stats_95105['le2500']} | {stats_95105['nan']} |
| **Total** | **{boundary_total}** | **{stats_1525['gt2500']+stats_4555['gt2500']+stats_95105['gt2500']}** | **{stats_1525['le2500']+stats_4555['le2500']+stats_95105['le2500']}** | **{stats_1525['nan']+stats_4555['nan']+stats_95105['nan']}** |

Tidak ditemukan masalah struktural yang menghambat proses labeling. Dataset dinyatakan **VALID**, proses dilanjutkan ke tahap labeling.

---

## 2. Distribusi `risk_label`

| risk_label | Jumlah | Persentase |
|---|---|---|
{rl_rows}
| **Total** | **{fmt_int(n_output)}** | **100%** |

---

## 3. Distribusi `label_source`

| label_source | Jumlah | Persentase |
|---|---|---|
{ls_rows}
| **Total** | **{fmt_int(n_output)}** | **100%** |

Rincian `RR_CAPE_MODIFIED` per boundary zone: {stats_1525['gt2500']} pada zona 15-25 mm, {stats_4555['gt2500']} pada zona 45-55 mm, {stats_95105['gt2500']} pada zona 95-105 mm.
Rincian `RR_ONLY_CAPE_MISSING` per boundary zone: {stats_1525['nan']} pada zona 15-25 mm, {stats_4555['nan']} pada zona 45-55 mm, {stats_95105['nan']} pada zona 95-105 mm.

---

## 4. Jumlah Record yang Dimodifikasi oleh CAPE

**{n_examples} record** mengalami kenaikan kategori risiko satu tingkat (label_source = `RR_CAPE_MODIFIED`), sesuai Aturan R3 (Tabel 3.5 BAB III): RR berada pada boundary zone dan CAPE > 2500 J/kg.

Rincian arah kenaikan:
{direction_lines}

---

## 5. Contoh Record yang Mengalami Modifikasi Label

| # | date | rr | cape | risk_label (final) | label_source |
|---|---|---|---|---|---|
{example_rows}

{example_note}

---

## 6. Validasi Integritas Data

| Pemeriksaan | Nilai |
|---|---|
| Row input | {fmt_int(integrity['row_input'])} |
| Row output | {fmt_int(integrity['row_output'])} |
| Selisih baris | {integrity['selisih_baris']} |
| Jumlah `risk_label` kosong (NaN) | {integrity['risk_label_kosong']} |
| Jumlah `label_source` kosong (NaN) | {integrity['label_source_kosong']} |
| Jumlah baris terhapus | 0 |
| Jumlah nilai fitur asli yang diubah | 0 (tidak ada imputasi/scaling/balancing tambahan dilakukan) |

---

## Acceptance Criteria

- [x] Tidak ada label kosong
- [x] Tidak ada label_source kosong
- [x] Tidak ada record yang dihapus
- [x] Row output ({fmt_int(n_output)}) = Row input ({fmt_int(n_input)})

## **STAGE 8 = PASS**
"""

with open(OUTPUT_REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write(report)

print(f"Laporan disimpan ke: {OUTPUT_REPORT_PATH}")


Laporan disimpan ke: /mnt/user-data/outputs/STAGE8_REPORT.md


In [22]:
print("=== RINGKASAN AKHIR TAHAP 8 ===")
print(f"Row input       : {n_input}")
print(f"Row output      : {n_output}")
print(f"risk_label kosong    : {int(df['risk_label'].isna().sum())}")
print(f"label_source kosong  : {int(df['label_source'].isna().sum())}")
print()
print("STAGE 8 = PASS" if (n_input == n_output and df['risk_label'].isna().sum() == 0 and df['label_source'].isna().sum() == 0) else "STAGE 8 = FAIL")


=== RINGKASAN AKHIR TAHAP 8 ===
Row input       : 2922
Row output      : 2922
risk_label kosong    : 0
label_source kosong  : 0

STAGE 8 = PASS
